# Extraction Walkthrough — Synthetic Fixture

Drives the docling-graph `/extract-pass` endpoint with a hand-crafted 2-paragraph fixture, prints the **exact request** sent to the service, the **raw response** (including the library log, which captures the LLM's structured output), and rolls up entities + fields + relationships at the end.

Unlike `ingest_walkthrough.ipynb` this notebook does **not** touch Postgres / MinIO / ArcadeDB — it's a pure side-channel against the running docling-graph container. Safe to run while the production worker is processing other documents.

**Prerequisite:** docling-graph running on `localhost:8002` (the default in `docker-compose.yml`). Verify with `curl http://localhost:8002/health`.

Active bundle: `air_defense_v3`. Field-group passes used here:
1. `radar_identity` — extracts RADAR_SYSTEM identity (system_name, type, etc.)
2. `radar_power_rf` — extracts power/RF numeric parameters per system
3. `radar_antenna` — extracts antenna parameters
4. `system_links` — extracts inter-entity relationships


## §1 Configuration + fixture text

The fake fixture mimics analyst prose: two distinct radar systems described with explicit numeric parameters that the field-group schema knows how to bind.

In [1]:
import json
import time
import urllib.request
from pprint import pprint

# Inside the Jupyter container we resolve the service via its docker network
# DNS name. From the host you'd use http://localhost:8002 instead.
DOCLING_GRAPH_BASE = "http://docling-graph:8002"
DOCLING_GRAPH_URL = f"{DOCLING_GRAPH_BASE}/extract-pass"
BUNDLE_KEY = "air_defense_v3"

# ─── FORCE_JSON_MODE flag ───────────────────────────────────────────────
# Toggle this to compare strict schema-grammar mode (False) vs loose JSON
# mode (True) on the same fixture. gemma4:31b reliably handles loose JSON
# but fails strict-grammar with unterminated-string parse errors on
# field-rich field-group schemas (e.g. radar_power_rf on parameter-heavy
# documents).
#
# This is a docling-graph SERVICE-LEVEL setting read at startup, so the
# notebook can't change it per-call. Set this flag, then run §1b below
# to get the exact host command that applies the flag and restarts
# docling-graph.
FORCE_JSON_MODE = True  # ← set True to test loose-JSON fallback

FAKE_TEXT = (
    # ── Radar paragraph 1 ────────────────────────────────────────────────
    "The Patriot AN/MPQ-65 is a multi-function phased-array fire-control "
    "radar deployed by the U.S. Army with the Patriot air-defense system. "
    "It operates in C-band at a nominal carrier frequency of 5500 MHz, "
    "with a peak transmitter output of 750 kW and a 35 dBi peak antenna "
    "gain. The array provides a 1.5 degree azimuth beamwidth.\n"
    "\n"
    # ── Radar paragraph 2 ────────────────────────────────────────────────
    "By contrast, the AN/SPY-6(V)1 air and missile defense radar (AMDR) "
    "operates in S-band around 3300 MHz with a peak transmit power of "
    "approximately 1500 kW. Its active electronically scanned array "
    "delivers 42 dBi gain with a 0.9 degree beamwidth and is integrated "
    "into the Aegis Combat System on Flight III destroyers.\n"
    "\n"
    # ── Missile paragraph 1 ──────────────────────────────────────────────
    "The MIM-104F Patriot Advanced Capability-3 (PAC-3) is the missile "
    "interceptor paired with the AN/MPQ-65 fire-control radar and is "
    "operational with the U.S. Army. The PAC-3 has a body length of "
    "5.2 meters and a body diameter of 0.255 meters, with a total launch "
    "mass of 316 kg. It achieves a maximum intercept range of 35 km and "
    "engages targets up to 25 km altitude, with a minimum engagement "
    "altitude of 0.05 km. The missile uses active radar homing guidance "
    "and reaches Mach 5 (approximately 1700 m/s). Its single-stage solid "
    "rocket booster produces 100 kN of thrust over a 2.5-second burn.\n"
    "\n"
    # ── Missile paragraph 2 ──────────────────────────────────────────────
    "The RIM-174 Standard Missile 6 (SM-6) Block IA is the air and "
    "missile defense interceptor paired with the AN/SPY-6(V)1 AESA radar "
    "on Aegis-equipped destroyers. The SM-6 has a body length of 6.55 "
    "meters and a diameter of 0.34 meters with a total launch mass of "
    "1500 kg. Maximum intercept range exceeds 240 km. The Mark 72 "
    "booster delivers approximately 290 kN thrust over a 6-second burn, "
    "after which the dual-pulse Mark 104 sustainer provides extended "
    "cruise. The SM-6 employs semi-active radar homing with terminal "
    "active radar guidance and achieves Mach 3.5 (approximately 1190 "
    "m/s)."
)

print(f"FORCE_JSON_MODE flag = {FORCE_JSON_MODE}")
print()
print(FAKE_TEXT)


FORCE_JSON_MODE flag = True

The Patriot AN/MPQ-65 is a multi-function phased-array fire-control radar deployed by the U.S. Army with the Patriot air-defense system. It operates in C-band at a nominal carrier frequency of 5500 MHz, with a peak transmitter output of 750 kW and a 35 dBi peak antenna gain. The array provides a 1.5 degree azimuth beamwidth.

By contrast, the AN/SPY-6(V)1 air and missile defense radar (AMDR) operates in S-band around 3300 MHz with a peak transmit power of approximately 1500 kW. Its active electronically scanned array delivers 42 dBi gain with a 0.9 degree beamwidth and is integrated into the Aegis Combat System on Flight III destroyers.

The MIM-104F Patriot Advanced Capability-3 (PAC-3) is the missile interceptor paired with the AN/MPQ-65 fire-control radar and is operational with the U.S. Army. The PAC-3 has a body length of 5.2 meters and a body diameter of 0.255 meters, with a total launch mass of 316 kg. It achieves a maximum intercept range of 35 km a

## §1b Apply the `FORCE_JSON_MODE` flag

`DOCLING_GRAPH_FORCE_JSON_MODE` is a service-level env var read once at startup, so flipping the flag above requires restarting docling-graph with the new value. Run the cell below to get the exact host command.

After running the host command, wait ~10s for the service to come back healthy, then continue with §2.

In [2]:
target_value = "true" if FORCE_JSON_MODE else "false"

# Best-effort: probe the running docling-graph to see whether the requested
# value is already in effect. The /health endpoint doesn't expose env, so
# we just confirm the service is reachable and let the operator verify
# DOCLING_GRAPH_FORCE_JSON_MODE in the rebuilt container's env.
try:
    with urllib.request.urlopen(f"{DOCLING_GRAPH_BASE}/health", timeout=5) as r:
        h = json.loads(r.read())
    print(f"docling-graph reachable (schema_count={h.get('schema_count')}, "
          f"pipeline_version={h.get('pipeline_version')})")
except Exception as exc:
    print(f"docling-graph NOT reachable: {exc}")

print()
print("=" * 72)
print(f"Apply FORCE_JSON_MODE={FORCE_JSON_MODE} — run from your HOST shell:")
print("=" * 72)
print()
print(f"  DOCLING_GRAPH_FORCE_JSON_MODE={target_value} \\")
print(f"      docker compose --profile split up -d --no-build "
      f"--force-recreate docling-graph")
print()
print("Verify the service picked up the new value:")
print()
print(f"  docker exec eip-mmdpp-docling-graph-1 env | "
      f"grep DOCLING_GRAPH_FORCE_JSON_MODE")
print()
print(f"Expected output: DOCLING_GRAPH_FORCE_JSON_MODE={target_value}")

docling-graph reachable (schema_count=12, pipeline_version=1.5.0)

Apply FORCE_JSON_MODE=True — run from your HOST shell:

  DOCLING_GRAPH_FORCE_JSON_MODE=true \
      docker compose --profile split up -d --no-build --force-recreate docling-graph

Verify the service picked up the new value:

  docker exec eip-mmdpp-docling-graph-1 env | grep DOCLING_GRAPH_FORCE_JSON_MODE

Expected output: DOCLING_GRAPH_FORCE_JSON_MODE=true


## §2 Build the DoclingDocument JSON

This is the **exact payload shape** the production worker sends. The DoclingDocument is the canonical structured form of any input document — even our fake plain text gets wrapped in this envelope. The LLM never sees this raw JSON; docling-graph chunks it into ~512-token pieces and feeds those to the model. But this is the input from the worker's point of view.

In [3]:
def build_docling_document(text: str, name: str = "synthetic-fixture") -> dict:
    """Minimal valid DoclingDocument with one text paragraph.

    Mirrors what app/workers/pipeline.py:_build_extract_pass_request sends
    when a real PDF has been Docling-parsed — except for our fake input
    we synthesize one `texts[]` entry instead of the dozens-to-hundreds
    that come out of OCR.
    """
    return {
        "schema_name": "DoclingDocument",
        "version": "1.0.0",
        "name": name,
        "origin": {
            "mimetype": "text/plain",
            "binary_hash": 1,
            "filename": "smoke.txt",
        },
        "furniture": {"name": "_root_", "self_ref": "#/furniture", "children": []},
        "body": {
            "name": "_root_",
            "self_ref": "#/body",
            "children": [{"$ref": "#/texts/0"}],
        },
        "groups": [],
        "pictures": [],
        "tables": [],
        "key_value_items": [],
        "form_items": [],
        "pages": {},
        "texts": [{
            "self_ref": "#/texts/0",
            "parent": {"$ref": "#/body"},
            "label": "text",
            "prov": [],
            "orig": text,
            "text": text,
        }],
    }

doc = build_docling_document(FAKE_TEXT)
print(json.dumps(doc, indent=2))

{
  "schema_name": "DoclingDocument",
  "version": "1.0.0",
  "name": "synthetic-fixture",
  "origin": {
    "mimetype": "text/plain",
    "binary_hash": 1,
    "filename": "smoke.txt"
  },
  "furniture": {
    "name": "_root_",
    "self_ref": "#/furniture",
    "children": []
  },
  "body": {
    "name": "_root_",
    "self_ref": "#/body",
    "children": [
      {
        "$ref": "#/texts/0"
      }
    ]
  },
  "groups": [],
  "pictures": [],
  "tables": [],
  "key_value_items": [],
  "form_items": [],
  "pages": {},
  "texts": [
    {
      "self_ref": "#/texts/0",
      "parent": {
        "$ref": "#/body"
      },
      "label": "text",
      "prov": [],
      "orig": "The Patriot AN/MPQ-65 is a multi-function phased-array fire-control radar deployed by the U.S. Army with the Patriot air-defense system. It operates in C-band at a nominal carrier frequency of 5500 MHz, with a peak transmitter output of 750 kW and a 35 dBi peak antenna gain. The array provides a 1.5 degree azimuth

## §3 Helper — POST to /extract-pass and capture everything

This wraps the HTTP call. Returns the request body (so you can see exactly what was sent) and the full response. On the server side this triggers:

```
main.py:extract_pass(...)
  → load_bundle_manifest("air_defense_v3")          # find pass by name
  → load_pass_template(...)                          # Pydantic model
  → docling_graph.run_pipeline(...)                  # internal LLM fan-out
    → DocumentChunker.chunk(doc, max_tokens=512)
    → for each batch: LiteLLM.call(model=gemma4:31b, schema=template)
  → graph_to_pass_output(...)                        # node-link → pass_output
  → service_postprocess + identity_gate + evidence_gate
  → ExtractPassResponse
```

In [4]:
def call_extract_pass(
    pass_name: str,
    document: dict,
    *,
    upstream_entities: list[dict] | None = None,
    timeout: int = 7200,
):
    """POST one /extract-pass call. Returns (request_body, response_dict, elapsed_seconds).

    `upstream_entities` is required for `document_plus_entity_refs` passes
    (e.g. system_links). The expected shape mirrors what the production
    worker sends from app/workers/pipeline.py:_build_extract_pass_request:

        [
            {"ref_id": "<arbitrary stable id>",
             "entity_type": "RADAR_SYSTEM" | "MISSILE_SYSTEM" | ...,
             "identity_values": {"system_name": "AN/MPQ-65", ...},
             "display_label": "AN/MPQ-65"},
            ...
        ]
    """
    request_body = {
        "bundle_key": BUNDLE_KEY,
        "pass_name": pass_name,
        "document_id": f"notebook-{pass_name}",
        "docling_document_json": document,
    }
    if upstream_entities:
        request_body["upstream_entities"] = upstream_entities
    payload = json.dumps(request_body).encode()
    req = urllib.request.Request(
        DOCLING_GRAPH_URL,
        data=payload,
        headers={"Content-Type": "application/json"},
    )
    t0 = time.monotonic()
    with urllib.request.urlopen(req, timeout=timeout) as r:
        response = json.loads(r.read())
    elapsed = time.monotonic() - t0
    return request_body, response, elapsed

# Quick health check
with urllib.request.urlopen(f"{DOCLING_GRAPH_BASE}/health", timeout=5) as r:
    print(json.loads(r.read()))


{'status': 'ok', 'schema_count': 12, 'extraction_contract': 'delta', 'pipeline_version': '1.5.0'}


## §3b Inspect what the LLM actually sees

The `/extract-pass` API call is the *outer* envelope. Inside docling-graph, the request gets fanned out into one or more LLM calls. Each LLM call gets:

1. A **SYSTEM prompt** — the global instructions (rules for emitting nodes, identity validation, FORBIDDEN-name policy, etc.). Source-of-truth: `ontology_bundles/_shared/prompt_rules.py::DELTA_SYSTEM_PROMPT` — overridden into the library prompt at request time.
2. A **USER prompt** — assembled per batch from:
   - The `path_catalog_block` (Pydantic-derived field catalog with descriptions)
   - The `schema_semantic_guide` (per-field guidance generated from the schema)
   - The `batch_markdown` (one or more chunks of the doc, format-rendered)
   - Optional `global_context` (first chunk preview) and `already_found` (prior-batch entities)
3. A **format= directive** to Ollama:
   - `format="json"` (loose) when `DOCLING_GRAPH_FORCE_JSON_MODE=true` OR the schema is over the 20 KB threshold
   - `format=<sanitized JSON Schema>` (strict grammar) otherwise — Ollama enforces JSON-Schema-conforming output token-by-token

The cell below defines `inspect_llm_prompt(pass_name)` — it reproduces docling-graph's internal assembly using the *same* library functions, without making an HTTP call. Each pass cell that follows calls it before the actual API call, so you can see exactly what's being asked of the LLM.

In [5]:
"""Helper that reconstructs the LLM prompt + schema for any pass.

Mirrors docling-graph's internal assembly path
(docker/docling-graph/app/main.py and the docling-graph library).
"""
from importlib import import_module
from docling_core.types.doc import DoclingDocument
from docling_graph.core.extractors.document_chunker import DocumentChunker
from docling_graph.core.extractors.contracts.delta.helpers import chunk_batches_by_token_limit
from docling_graph.core.extractors.contracts.delta.catalog import build_delta_node_catalog
from docling_graph.core.extractors.contracts.delta.schema_mapper import (
    build_catalog_prompt_block, build_delta_semantic_guide,
)
from docling_graph.core.extractors.contracts.delta.prompts import (
    get_delta_batch_prompt, format_batch_markdown,
)
# Service-side rewrite: the docling-graph service replaces the library's
# default system prompt with the source-of-truth in
# ontology_bundles/_shared/prompt_rules.py. Match that here.
from ontology_bundles._shared.prompt_rules import DELTA_SYSTEM_PROMPT

# All 12 active passes from manifest.yaml. Mirrors PASS_MODULES from the
# ingest_walkthrough notebook so this notebook can stand alone.
PASS_MODULES = {
    "radar_identity":       ("ontology_bundles.air_defense_v3.extraction_schemas.radar_identity",       "RadarIdentityPass"),
    "radar_power_rf":       ("ontology_bundles.air_defense_v3.extraction_schemas.radar_power_rf",       "RadarPowerRfPass"),
    "radar_antenna":        ("ontology_bundles.air_defense_v3.extraction_schemas.radar_antenna",        "RadarAntennaPass"),
    "radar_timing":         ("ontology_bundles.air_defense_v3.extraction_schemas.radar_timing",         "RadarTimingPass"),
    "radar_modulation":     ("ontology_bundles.air_defense_v3.extraction_schemas.radar_modulation",     "RadarModulationPass"),
    "missile_identity":     ("ontology_bundles.air_defense_v3.extraction_schemas.missile_identity",     "MissileIdentityPass"),
    "missile_kinematics":   ("ontology_bundles.air_defense_v3.extraction_schemas.missile_kinematics",   "MissileKinematicsPass"),
    "missile_guidance":     ("ontology_bundles.air_defense_v3.extraction_schemas.missile_guidance",     "MissileGuidancePass"),
    "missile_airframe":     ("ontology_bundles.air_defense_v3.extraction_schemas.missile_airframe",     "MissileAirframePass"),
    "missile_speed_timing": ("ontology_bundles.air_defense_v3.extraction_schemas.missile_speed_timing", "MissileSpeedTimingPass"),
    "missile_propulsion":   ("ontology_bundles.air_defense_v3.extraction_schemas.missile_propulsion",   "MissilePropulsionPass"),
    "system_links":         ("ontology_bundles.air_defense_v3.extraction_schemas.system_links",         "SystemLinksPass"),
}

# Cap big sections so output stays scannable; flip these to None to dump everything.
SCHEMA_PRINT_CAP = None
USER_PROMPT_CAP  = None


def inspect_llm_prompt(pass_name, document=None, *, batch_index=0):
    """Print the SYSTEM + USER prompts, JSON Schema, and format-mode for one pass."""
    if pass_name not in PASS_MODULES:
        raise ValueError(f"Unknown pass_name: {pass_name!r}; choices: {list(PASS_MODULES)}")
    if document is None:
        document = doc

    mod_path, cls_name = PASS_MODULES[pass_name]
    template_cls = getattr(import_module(mod_path), cls_name)

    # 1. Chunk + batch the doc the same way production does
    docling_doc = DoclingDocument.model_validate(document)
    chunker = DocumentChunker(
        tokenizer_name="sentence-transformers/all-MiniLM-L6-v2",
        chunk_max_tokens=512,
        merge_peers=True,
    )
    chunks = chunker.chunk_document(docling_doc)
    token_counts = [chunker.tokenizer.count_tokens(c) for c in chunks]
    batch_plan = chunk_batches_by_token_limit(chunks, token_counts, max_batch_tokens=1024)

    if batch_index >= len(batch_plan):
        print(f"batch_index={batch_index} out of range (only {len(batch_plan)} batches)")
        return

    selected_batch = batch_plan[batch_index]
    batch_texts = [text for _idx, text, _tok in selected_batch]
    batch_markdown = format_batch_markdown(batch_texts)

    # 2. Build catalog + semantic guide from the Pydantic template
    catalog        = build_delta_node_catalog(template_cls)
    catalog_block  = build_catalog_prompt_block(catalog)
    schema_dict    = template_cls.model_json_schema()
    semantic_guide = build_delta_semantic_guide(template_cls, schema_dict)

    first_chunk = chunks[0].strip() if chunks else ""
    global_context = (first_chunk[:600] + ("..." if len(first_chunk) > 600 else "")) if first_chunk else None

    # 3. Compose the prompt; override the library's system prompt with the
    #    service-side source-of-truth (matches docling-graph/app/main.py).
    _orig = get_delta_batch_prompt
    def _patched(**kw):
        r = _orig(**kw)
        if isinstance(r, dict) and "system" in r:
            r["system"] = DELTA_SYSTEM_PROMPT
        return r

    prompt = _patched(
        batch_markdown=batch_markdown,
        schema_semantic_guide=semantic_guide,
        path_catalog_block=catalog_block,
        batch_index=batch_index,
        total_batches=len(batch_plan),
        global_context=global_context,
        already_found=None,
    )

    # 4. Determine format mode the service will use for this call
    schema_str = json.dumps(schema_dict)
    threshold  = 20000  # docking-graph config_builder default
    if FORCE_JSON_MODE:
        format_mode = f"json (loose)  ← FORCE_JSON_MODE=true"
    elif len(schema_str) > threshold:
        format_mode = f"json (loose)  ← schema {len(schema_str)} chars > {threshold} threshold"
    else:
        format_mode = f"<schema> (strict grammar; schema {len(schema_str)} chars)"

    # 5. Render
    print(f"========== {pass_name}  template={cls_name} ==========")
    print(f"chunks={len(chunks)}  total_tokens={sum(token_counts)}  batches={len(batch_plan)}  batch_index={batch_index}")
    print(f"format_mode → {format_mode}")
    print(f"catalog_paths={len(catalog.nodes)}  semantic_guide_chars={len(semantic_guide)}")
    print()
    print("--- SYSTEM PROMPT ---")
    print(prompt["system"])
    print()
    print("--- USER PROMPT ---")
    user = prompt["user"]
    print(user[:USER_PROMPT_CAP] + ("\n[...truncated]" if USER_PROMPT_CAP and len(user) > USER_PROMPT_CAP else ""))
    print()
    print("--- JSON SCHEMA (template_cls.model_json_schema()) ---")
    sd = json.dumps(schema_dict, indent=2)
    print(sd[:SCHEMA_PRINT_CAP] + ("\n[...truncated]" if SCHEMA_PRINT_CAP and len(sd) > SCHEMA_PRINT_CAP else ""))


# Sanity check: list pass names
print("inspect_llm_prompt() ready. Pass names:")
for p in PASS_MODULES:
    print(f"  - {p}")

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


inspect_llm_prompt() ready. Pass names:
  - radar_identity
  - radar_power_rf
  - radar_antenna
  - radar_timing
  - radar_modulation
  - missile_identity
  - missile_kinematics
  - missile_guidance
  - missile_airframe
  - missile_speed_timing
  - missile_propulsion
  - system_links


## §4 Pass 1 — `radar_identity`

Extracts the high-level identity for each radar mentioned in the text: `system_name`, `nomenclature`, `radar_type`, `band`, etc. — the *who* layer. Subsequent passes populate per-system numeric fields against these identities.

In [6]:
# What the LLM is asked
inspect_llm_prompt("radar_identity")
print("\n" + "=" * 72 + "\n")

# Actual API call
req_id, resp_id, t_id = call_extract_pass("radar_identity", doc)

print(f"-- elapsed: {t_id:.1f}s --\n")
print("-- complete request body --")
print(json.dumps(req_id, indent=2))
print()
print("-- response: pass_output.radar_systems --")
for system in resp_id.get("pass_output", {}).get("radar_systems", []):
    print(json.dumps(system, indent=2))

[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== radar_identity  template=RadarIdentityPass ==========
chunks=1  total_tokens=449  batches=1  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=2219

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Field Guidance** 

In [7]:
# Service-level diagnostics — what docling-graph saw internally
diags = resp_id.get("diagnostics", {})
print("chunk_count:", diags.get("chunk_count"))
print("batch_count:", diags.get("batch_count"))
print("parallel_workers:", diags.get("parallel_workers"))
print("llm_batch_token_size:", diags.get("llm_batch_token_size"))
print("path_counts:", diags.get("path_counts"))
print("quality_gate.ok:", diags.get("quality_gate", {}).get("ok"))
print("quality_gate.reasons:", diags.get("quality_gate", {}).get("reasons"))
print()
print("-- batch_timings (per-LLM-call elapsed) --")
for bt in diags.get("batch_timings", []):
    print(f"  batch {bt.get('batch_index')}: {bt.get('elapsed_seconds'):.2f}s")

chunk_count: 1
batch_count: 1
parallel_workers: 5
llm_batch_token_size: 1024
path_counts: {'': 1, 'radar_systems[]': 2}
quality_gate.ok: True
quality_gate.reasons: []

-- batch_timings (per-LLM-call elapsed) --
  batch 0: 18.37s


In [8]:
# library_log captures the docling-graph library's stdout during this call —
# this is the closest thing to seeing the LLM's raw JSON output (post-parse).
# If you set DOCLING_GRAPH_FORCE_JSON_MODE=true and restart the service,
# the warning lines about 'Structured output failed' / 'retrying with legacy
# prompt-schema mode' should disappear here.
print(diags.get("library_log", ""))

[LlmBackend] Initialized with:
  • Client: LiteLLMClient
  • Model: gemma4:31b
tokenizer_config.json: 100%|##########| 350/350 [00:00<00:00, 1.90MB/s]
vocab.txt: 232kB [00:00, 28.3MB/s]
tokenizer.json: 466kB [00:00, 57.8MB/s]
special_tokens_map.json: 100%|##########| 112/112 [00:00<00:00, 1.86MB/s]
[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True
[DocumentProcessor] Initialized with Classic OCR pipeline (English, French)
[ExtractorFactory] Created ManyToOneStrategy
[DeltaExtraction] Delta extraction from DoclingDocument (1824 chars)
[DeltaExtraction] Calling LLM...
[LlmBackend] Successfully extracted data from DoclingDocument
[GraphConverter] Pre-registering models for deterministic node IDs...
[GraphConverter] Running automatic graph cleanup...
[GraphCleaner] Starting cleanup: 3 nodes, 2 edges
[GraphCleaner] Cleanup complete:
  • Removed 0 phantom nodes
  • Merged 0 duplicate nodes
  • Removed 0 s

## §5 Pass 2 — `radar_power_rf`

Per-system numeric fields: `nominal_rf_mhz`, `tx_peak_power_kw`, `prf_hz`, `pulse_width_us`, etc. The LLM has to bind the same `system_name` it used in `radar_identity` (free-form prompt — there is no formal `upstream_entities` for `document_only` field-group passes; identity is pinned by the prompt template referencing `system_name` as the primary key).

In [9]:
# What the LLM is asked
inspect_llm_prompt("radar_power_rf")
print("\n" + "=" * 72 + "\n")

# Actual API call
req_pr, resp_pr, t_pr = call_extract_pass("radar_power_rf", doc)

print(f"-- elapsed: {t_pr:.1f}s --\n")
print("-- complete request body --")
print(json.dumps(req_pr, indent=2))
print()
print("-- response: pass_output.radar_systems --")
for system in resp_pr.get("pass_output", {}).get("radar_systems", []):
    populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
    print(
        f"{system.get('system_name'):<25} "
        f"populated_fields={len(populated)}: {sorted(populated.keys())}"
    )
    print(json.dumps(populated, indent=2))
    print()

[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== radar_power_rf  template=RadarPowerRfPass ==========
chunks=1  total_tokens=449  batches=1  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=940

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Field Guidance** as

## §6 Pass 3 — `radar_antenna`

Antenna parameters: `gain_dbi`, `azimuth_beamwidth_deg`, `elevation_beamwidth_deg`, `aperture_m`, scan-type enum, etc.

In [10]:
# What the LLM is asked
inspect_llm_prompt("radar_antenna")
print("\n" + "=" * 72 + "\n")

# Actual API call
req_an, resp_an, t_an = call_extract_pass("radar_antenna", doc)

print(f"-- elapsed: {t_an:.1f}s --\n")
print("-- complete request body --")
print(json.dumps(req_an, indent=2))
print()
print("-- response: pass_output.radar_systems --")
for system in resp_an.get("pass_output", {}).get("radar_systems", []):
    populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
    print(json.dumps(populated, indent=2))
    print()

[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== radar_antenna  template=RadarAntennaPass ==========
chunks=1  total_tokens=449  batches=1  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=1498

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Field Guidance** as

## §7 `radar_timing`

Field-group pass — populates `radar_systems[]` with the per-system fields owned by this group. See `ontology_bundles/air_defense_v3/extraction_schemas/radar_timing.py` for the field list.

In [11]:
# What the LLM is asked
inspect_llm_prompt("radar_timing")
print("\n" + "=" * 72 + "\n")

# Actual API call
req_rt, resp_rt, t_rt = call_extract_pass("radar_timing", doc)

print(f"-- elapsed: {t_rt:.1f}s --\n")
print("-- complete request body --")
print(json.dumps(req_rt, indent=2))
print()
print("-- response: pass_output.radar_systems --")
for system in resp_rt.get("pass_output", {}).get("radar_systems", []) or []:
    populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
    print(json.dumps(populated, indent=2))
    print()


[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== radar_timing  template=RadarTimingPass ==========
chunks=1  total_tokens=449  batches=1  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=1132

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Field Guidance** as t

## §8 `radar_modulation`

Field-group pass — populates `radar_systems[]` with the per-system fields owned by this group. See `ontology_bundles/air_defense_v3/extraction_schemas/radar_modulation.py` for the field list.

In [12]:
# What the LLM is asked
inspect_llm_prompt("radar_modulation")
print("\n" + "=" * 72 + "\n")

# Actual API call
req_rm, resp_rm, t_rm = call_extract_pass("radar_modulation", doc)

print(f"-- elapsed: {t_rm:.1f}s --\n")
print("-- complete request body --")
print(json.dumps(req_rm, indent=2))
print()
print("-- response: pass_output.radar_systems --")
for system in resp_rm.get("pass_output", {}).get("radar_systems", []) or []:
    populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
    print(json.dumps(populated, indent=2))
    print()


[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== radar_modulation  template=RadarModulationPass ==========
chunks=1  total_tokens=449  batches=1  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=1435

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Field Guidanc

## §9 `missile_identity`

Field-group pass — populates `missile_systems[]` with the per-system fields owned by this group. See `ontology_bundles/air_defense_v3/extraction_schemas/missile_identity.py` for the field list.

In [13]:
# What the LLM is asked
inspect_llm_prompt("missile_identity")
print("\n" + "=" * 72 + "\n")

# Actual API call
req_mi, resp_mi, t_mi = call_extract_pass("missile_identity", doc)

print(f"-- elapsed: {t_mi:.1f}s --\n")
print("-- complete request body --")
print(json.dumps(req_mi, indent=2))
print()
print("-- response: pass_output.missile_systems --")
for system in resp_mi.get("pass_output", {}).get("missile_systems", []) or []:
    populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
    print(json.dumps(populated, indent=2))
    print()


[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== missile_identity  template=MissileIdentityPass ==========
chunks=1  total_tokens=449  batches=1  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=1870

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Field Guidanc

## §10 `missile_kinematics`

Field-group pass — populates `missile_systems[]` with the per-system fields owned by this group. See `ontology_bundles/air_defense_v3/extraction_schemas/missile_kinematics.py` for the field list.

In [14]:
# What the LLM is asked
inspect_llm_prompt("missile_kinematics")
print("\n" + "=" * 72 + "\n")

# Actual API call
req_mk, resp_mk, t_mk = call_extract_pass("missile_kinematics", doc)

print(f"-- elapsed: {t_mk:.1f}s --\n")
print("-- complete request body --")
print(json.dumps(req_mk, indent=2))
print()
print("-- response: pass_output.missile_systems --")
for system in resp_mk.get("pass_output", {}).get("missile_systems", []) or []:
    populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
    print(json.dumps(populated, indent=2))
    print()


[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== missile_kinematics  template=MissileKinematicsPass ==========
chunks=1  total_tokens=449  batches=1  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=1237

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Field Gui

## §11 `missile_guidance`

Field-group pass — populates `missile_systems[]` with the per-system fields owned by this group. See `ontology_bundles/air_defense_v3/extraction_schemas/missile_guidance.py` for the field list.

In [15]:
# What the LLM is asked
inspect_llm_prompt("missile_guidance")
print("\n" + "=" * 72 + "\n")

# Actual API call
req_mg, resp_mg, t_mg = call_extract_pass("missile_guidance", doc)

print(f"-- elapsed: {t_mg:.1f}s --\n")
print("-- complete request body --")
print(json.dumps(req_mg, indent=2))
print()
print("-- response: pass_output.missile_systems --")
for system in resp_mg.get("pass_output", {}).get("missile_systems", []) or []:
    populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
    print(json.dumps(populated, indent=2))
    print()


[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== missile_guidance  template=MissileGuidancePass ==========
chunks=1  total_tokens=449  batches=1  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=1166

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Field Guidanc

## §12 `missile_airframe`

Field-group pass — populates `missile_systems[]` with the per-system fields owned by this group. See `ontology_bundles/air_defense_v3/extraction_schemas/missile_airframe.py` for the field list.

In [16]:
# What the LLM is asked
inspect_llm_prompt("missile_airframe")
print("\n" + "=" * 72 + "\n")

# Actual API call
req_ma, resp_ma, t_ma = call_extract_pass("missile_airframe", doc)

print(f"-- elapsed: {t_ma:.1f}s --\n")
print("-- complete request body --")
print(json.dumps(req_ma, indent=2))
print()
print("-- response: pass_output.missile_systems --")
for system in resp_ma.get("pass_output", {}).get("missile_systems", []) or []:
    populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
    print(json.dumps(populated, indent=2))
    print()


[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== missile_airframe  template=MissileAirframePass ==========
chunks=1  total_tokens=449  batches=1  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=955

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Field Guidance

## §13 `missile_speed_timing`

Field-group pass — populates `missile_systems[]` with the per-system fields owned by this group. See `ontology_bundles/air_defense_v3/extraction_schemas/missile_speed_timing.py` for the field list.

In [17]:
# What the LLM is asked
inspect_llm_prompt("missile_speed_timing")
print("\n" + "=" * 72 + "\n")

# Actual API call
req_mst, resp_mst, t_mst = call_extract_pass("missile_speed_timing", doc)

print(f"-- elapsed: {t_mst:.1f}s --\n")
print("-- complete request body --")
print(json.dumps(req_mst, indent=2))
print()
print("-- response: pass_output.missile_systems --")
for system in resp_mst.get("pass_output", {}).get("missile_systems", []) or []:
    populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
    print(json.dumps(populated, indent=2))
    print()


[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== missile_speed_timing  template=MissileSpeedTimingPass ==========
chunks=1  total_tokens=449  batches=1  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=1580

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Field 

## §14 `missile_propulsion`

Field-group pass — populates `missile_systems[]` with the per-system fields owned by this group. See `ontology_bundles/air_defense_v3/extraction_schemas/missile_propulsion.py` for the field list.

In [18]:
# What the LLM is asked
inspect_llm_prompt("missile_propulsion")
print("\n" + "=" * 72 + "\n")

# Actual API call
req_mp, resp_mp, t_mp = call_extract_pass("missile_propulsion", doc)

print(f"-- elapsed: {t_mp:.1f}s --\n")
print("-- complete request body --")
print(json.dumps(req_mp, indent=2))
print()
print("-- response: pass_output.missile_systems --")
for system in resp_mp.get("pass_output", {}).get("missile_systems", []) or []:
    populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
    print(json.dumps(populated, indent=2))
    print()


[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== missile_propulsion  template=MissilePropulsionPass ==========
chunks=1  total_tokens=449  batches=1  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=2027

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Field Gui

## §15 `system_links` (relationships)

Cross-domain relationship pass. Now we DO have upstream entities (from `radar_identity` + `missile_identity`), so we pass them to the service so it can emit relationship edges (`ASSOCIATED_WITH` / `CUES`) between the radars and missiles in the prose.

In [19]:
# Build upstream_entities from radar_identity + missile_identity outputs.
# Mirrors what app/workers/pipeline.py:_run_single_pass does in production.
upstream_entities = []
for entity in resp_id.get("pass_output", {}).get("radar_systems", []) or []:
    name = entity.get("system_name")
    if name:
        upstream_entities.append({
            "ref_id":          f"RADAR_SYSTEM:{name}",
            "entity_type":     "RADAR_SYSTEM",
            "identity_values": {"system_name": name},
            "display_label":   name,
        })
for entity in resp_mi.get("pass_output", {}).get("missile_systems", []) or []:
    name = entity.get("system_name")
    if name:
        upstream_entities.append({
            "ref_id":          f"MISSILE_SYSTEM:{name}",
            "entity_type":     "MISSILE_SYSTEM",
            "identity_values": {"system_name": name},
            "display_label":   name,
        })

print(f"-- {len(upstream_entities)} upstream_entities pinned --")
for u in upstream_entities:
    print(f"  {u['entity_type']:<14} {u['system_name'] if 'system_name' in u else u['display_label']}")
print()

# What the LLM is asked
inspect_llm_prompt("system_links")
print("\n" + "=" * 72 + "\n")

# Actual API call (now with upstream_entities)
try:
    req_sl, resp_sl, t_sl = call_extract_pass(
        "system_links", doc, upstream_entities=upstream_entities,
    )
    print(f"-- elapsed: {t_sl:.1f}s --\n")
    print("-- complete request body --")
    print(json.dumps(req_sl, indent=2))
    print()
    print("-- response: pass_output --")
    print(json.dumps(resp_sl.get("pass_output", {}), indent=2))
except urllib.error.HTTPError as exc:
    print(f"HTTP {exc.code}: {exc.read().decode()[:500]}")
    resp_sl = {"pass_output": {}}


-- 4 upstream_entities pinned --
  RADAR_SYSTEM   AN/MPQ-65
  RADAR_SYSTEM   AN/SPY-6(V)1
  MISSILE_SYSTEM PAC-3
  MISSILE_SYSTEM SM-6



[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== system_links  template=SystemLinksPass ==========
chunks=1  total_tokens=449  batches=1  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=1689

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Field Guidance** as t

## §16 Rollup — entities × fields, edges

Combine outputs from all 11 entity-emitting passes. Production merges via `extraction_merge.merge_and_resolve()`; here we just zip pass outputs by `system_name` and split by entity type.

In [20]:
"""§16 — production-parity rollup.

Mirrors what app/services/extraction_merge.py:merge_and_resolve does:
  1. Phase 0 — canonicalize_cross_pass_identities: detect entities split
     across passes (e.g. missile_identity emits 'MIM-104F', missile_airframe
     emits 'PAC-3' from the same prose) and rewrite to a single canonical
     system_name BEFORE merging.
  2. Phase 1 — merge: zip by canonical system_name; union all populated fields.

This makes the notebook rollup match what would land in the production graph
after merge_and_resolve runs.
"""
import re
from collections import defaultdict


def _identity_token_bag(entity: dict) -> set[str]:
    """Tokenize system_name + nomenclature + name into UPPERCASE token set,
    dropping single-char tokens."""
    parts = [entity.get(f) for f in ("system_name", "nomenclature", "name")
             if isinstance(entity.get(f), str)]
    text = " ".join(p for p in parts if p)
    return {t.upper() for t in re.split(r"[^A-Za-z0-9]+", text) if len(t) >= 2}


def _name_only_tokens(entity: dict) -> set[str]:
    name = entity.get("system_name") or ""
    return {t.upper() for t in re.split(r"[^A-Za-z0-9]+", name) if len(t) >= 2}


def _likely_same_entity(a: dict, b: dict) -> bool:
    a_name, b_name = a.get("system_name"), b.get("system_name")
    if not a_name or not b_name:
        return False
    if a_name == b_name:
        return True
    a_tokens = _name_only_tokens(a)
    b_tokens = _name_only_tokens(b)
    if not a_tokens or not b_tokens:
        return False
    return a_tokens.issubset(_identity_token_bag(b)) or \
           b_tokens.issubset(_identity_token_bag(a))


def _pick_canonical(names):
    cands = sorted({n for n in names if isinstance(n, str) and n}, key=lambda n: (len(n), n))
    return cands[0] if cands else ""


def canonicalize_systems(per_pass_lists):
    flat = [e for sub in per_pass_lists for e in sub]
    if len(flat) <= 1:
        return
    n = len(flat)
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(x, y):
        rx, ry = find(x), find(y)
        if rx != ry:
            parent[rx] = ry
    for i in range(n):
        for j in range(i + 1, n):
            if _likely_same_entity(flat[i], flat[j]):
                union(i, j)
    components = defaultdict(list)
    for i in range(n):
        components[find(i)].append(i)
    rewrites = 0
    for member_idxs in components.values():
        if len(member_idxs) <= 1:
            continue
        canonical = _pick_canonical([flat[i].get("system_name") for i in member_idxs])
        if not canonical:
            continue
        for i in member_idxs:
            if flat[i].get("system_name") != canonical:
                flat[i]["system_name"] = canonical
                rewrites += 1
    if rewrites:
        print(f"canonicalize_systems: rewrote {rewrites} system_name(s) "
              f"to collapse cross-pass duplicates")


# ── Gather per-pass entity lists ────────────────────────────────────────
RADAR_PASSES = [
    ("identity",   resp_id),
    ("power_rf",   resp_pr),
    ("antenna",    resp_an),
    ("timing",     resp_rt),
    ("modulation", resp_rm),
]
MISSILE_PASSES = [
    ("identity",     resp_mi),
    ("kinematics",   resp_mk),
    ("guidance",     resp_mg),
    ("airframe",     resp_ma),
    ("speed_timing", resp_mst),
    ("propulsion",   resp_mp),
]

radar_lists   = [resp.get("pass_output", {}).get("radar_systems",   []) or []
                 for _, resp in RADAR_PASSES]
missile_lists = [resp.get("pass_output", {}).get("missile_systems", []) or []
                 for _, resp in MISSILE_PASSES]

# ── Phase 0 — cross-pass canonicalization (mirrors production) ──────────
canonicalize_systems(radar_lists)
canonicalize_systems(missile_lists)

# ── Phase 1 — merge by canonical system_name ────────────────────────────
merged_radar   = defaultdict(dict)
merged_missile = defaultdict(dict)

for (label, _), entities in zip(RADAR_PASSES, radar_lists):
    for system in entities:
        name = system.get("system_name")
        if not name:
            continue
        for k, v in system.items():
            if v not in (None, "", [], {}):
                merged_radar[name][k] = v
        merged_radar[name].setdefault("_passes_seen_in", set()).add(label)

for (label, _), entities in zip(MISSILE_PASSES, missile_lists):
    for system in entities:
        name = system.get("system_name")
        if not name:
            continue
        for k, v in system.items():
            if v not in (None, "", [], {}):
                merged_missile[name][k] = v
        merged_missile[name].setdefault("_passes_seen_in", set()).add(label)


def _print_block(label, merged):
    print(f"\n=========== {label} ===========")
    for name, fields in merged.items():
        passes = sorted(fields.pop("_passes_seen_in", set()))
        print(f"\n--- {name} (seen in: {', '.join(passes)}) ---")
        for k in sorted(fields.keys()):
            print(f"  {k:<28} = {fields[k]}")


_print_block("RADAR_SYSTEMS", merged_radar)
_print_block("MISSILE_SYSTEMS", merged_missile)



=========== RADAR_SYSTEMS ===========

--- AN/MPQ-65 (seen in: antenna, identity, modulation, power_rf, timing) ---
  beamwidth_az_deg             = 1.5
  gain_dbi                     = 35.0
  nominal_rf_mhz               = 5500.0
  scan_type                    = ELECTRONIC
  system_name                  = AN/MPQ-65
  system_status                = OPERATIONAL
  tx_peak_power_kw             = 750.0

--- AN/SPY-6(V)1 (seen in: antenna, identity, modulation, power_rf, timing) ---
  beamwidth_az_deg             = 0.9
  gain_dbi                     = 42.0
  nominal_rf_mhz               = 3300.0
  scan_type                    = ELECTRONIC
  system_name                  = AN/SPY-6(V)1
  tx_peak_power_kw             = 1500.0

=========== MISSILE_SYSTEMS ===========

--- PAC-3 (seen in: airframe, guidance, identity, kinematics, propulsion, speed_timing) ---
  body_diameter_m              = 0.255
  body_length_m                = 5.2
  booster_thrust               = 100 kN
  booster_time_sec   

In [21]:
# Cleaner tabular view (one DataFrame per entity type)
try:
    import pandas as pd
    df_radar = pd.DataFrame.from_dict(merged_radar, orient="index").fillna("-")
    df_missile = pd.DataFrame.from_dict(merged_missile, orient="index").fillna("-")
    print("=== RADAR_SYSTEMS ===")
    print(df_radar)
    print()
    print("=== MISSILE_SYSTEMS ===")
    print(df_missile)
except ImportError:
    print("pandas not installed; install with `pip install pandas`")


=== RADAR_SYSTEMS ===
               system_name system_status   scan_type  tx_peak_power_kw  \
AN/MPQ-65        AN/MPQ-65   OPERATIONAL  ELECTRONIC             750.0   
AN/SPY-6(V)1  AN/SPY-6(V)1             -  ELECTRONIC            1500.0   

              nominal_rf_mhz  gain_dbi  beamwidth_az_deg  
AN/MPQ-65             5500.0      35.0               1.5  
AN/SPY-6(V)1          3300.0      42.0               0.9  

=== MISSILE_SYSTEMS ===
      system_name                            nomenclature      name  \
PAC-3       PAC-3  MIM-104F Patriot Advanced Capability-3   Patriot   
SM-6         SM-6                                       -  Standard   

      system_status  max_intercept_km max_altitude_km  body_length_m  \
PAC-3   OPERATIONAL              35.0            25.0           5.20   
SM-6              -             240.0               -           6.55   

       body_diameter_m  total_mass_kg  max_speed_mps  total_burn_time_sec  \
PAC-3            0.255          316.0        

In [22]:
# Relationships emitted across all passes (system_links is the canonical source).
edges = []
for resp in (resp_id, resp_pr, resp_an, resp_rt, resp_rm,
             resp_mi, resp_mk, resp_mg, resp_ma, resp_mst, resp_mp, resp_sl):
    for edge in resp.get("pass_output", {}).get("relationships", []) or []:
        edges.append(edge)

if edges:
    print(f"-- {len(edges)} relationship(s) extracted --")
    for e in edges:
        print(json.dumps(e, indent=2))
else:
    print("No relationships extracted — system_links produced no edges.")


-- 2 relationship(s) extracted --
{
  "rel_type": "ASSOCIATED_WITH",
  "from_ref_id": "RADAR_SYSTEM:AN/MPQ-65",
  "to_ref_id": "MISSILE_SYSTEM:PAC-3",
  "confidence": 1.0
}
{
  "rel_type": "ASSOCIATED_WITH",
  "from_ref_id": "RADAR_SYSTEM:AN/SPY-6(V)1",
  "to_ref_id": "MISSILE_SYSTEM:SM-6",
  "confidence": 1.0
}


In [23]:
# Final summary
summary = {
    "radar_systems_extracted":   list(merged_radar.keys()),
    "missile_systems_extracted": list(merged_missile.keys()),
    "total_populated_radar_fields":   sum(len(f) for f in merged_radar.values()),
    "total_populated_missile_fields": sum(len(f) for f in merged_missile.values()),
    "per_pass_elapsed_seconds": {
        "radar_identity":       round(t_id, 1),
        "radar_power_rf":       round(t_pr, 1),
        "radar_antenna":        round(t_an, 1),
        "radar_timing":         round(t_rt, 1),
        "radar_modulation":     round(t_rm, 1),
        "missile_identity":     round(t_mi, 1),
        "missile_kinematics":   round(t_mk, 1),
        "missile_guidance":     round(t_mg, 1),
        "missile_airframe":     round(t_ma, 1),
        "missile_speed_timing": round(t_mst, 1),
        "missile_propulsion":   round(t_mp, 1),
        "system_links":         round(t_sl, 1) if "t_sl" in dir() else None,
    },
    "relationships": len(edges),
}
print(json.dumps(summary, indent=2))


{
  "radar_systems_extracted": [
    "AN/MPQ-65",
    "AN/SPY-6(V)1"
  ],
  "missile_systems_extracted": [
    "PAC-3",
    "SM-6"
  ],
  "total_populated_radar_fields": 13,
  "total_populated_missile_fields": 24,
  "per_pass_elapsed_seconds": {
    "radar_identity": 26.7,
    "radar_power_rf": 13.1,
    "radar_antenna": 17.9,
    "radar_timing": 9.6,
    "radar_modulation": 10.1,
    "missile_identity": 17.1,
    "missile_kinematics": 15.3,
    "missile_guidance": 13.3,
    "missile_airframe": 13.2,
    "missile_speed_timing": 19.7,
    "missile_propulsion": 19.9,
    "system_links": 18.7
  },
  "relationships": 2
}
